# Metal Visualization — Deep Exploration
## Group × Algorithm Correlations and Full Pairwise Analysis

---

**Scope:** Metal visualization only. Three groups with different programming experience levels:
- **Group 1** — No programming experience
- **Group 2** — Brief knowledge of programming
- **Group 3** — Multiple years of programming coursework

**Purpose:** Exhaustive correlation exploration across all metric pairs, with experience level as an additional ordinal variable. Find relationships that have instructional or design implications.

---
## 1. Setup & Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
from itertools import combinations
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')

ALG_PAL   = {'BFS': '#2166AC', 'DFS': '#D6604D'}
GRP_PAL   = {1: '#1B7837', 2: '#762A83', 3: '#E66101'}
GRP_PAL_S = {'1': '#1B7837', '2': '#762A83', '3': '#E66101'}  # string keys for seaborn
GRP_LABEL = {1: 'G1 (no exp)', 2: 'G2 (brief)', 3: 'G3 (years)'}
print('Setup complete.')

In [ ]:
METAL_FILES = [
    ('Group1_metalBFSData.xls', 'BFS', 'DE-bft.wmv', 1),
    ('Group1_metalDFSData.xls', 'DFS', 'DE-dft.wmv', 1),
    ('Group2_metalBFSData.xls', 'BFS', 'DE-bft.wmv', 2),
    ('Group2_metalDFSData.xls', 'DFS', 'DE-dft.wmv', 2),
    ('Group3_metalBFSData.xls', 'BFS', 'DE-bft.wmv', 3),
    ('Group3_metalDFSData.xls', 'DFS', 'DE-dft.wmv', 3),
]
STAT_KW = {'nan', 'mean', 'sum', 'std', 'median', '', 'all recordings'}

def get_col(df, metric, video, aoi):
    return next((c for c in df.columns
                 if metric in c and video in c and aoi in c
                 and c.endswith('_Mean') and 'Include Zeros' not in c), None)

records = []
for fname, algo, video, grp in METAL_FILES:
    path = DATA_DIR / fname
    df = pd.read_excel(path, engine='xlrd')
    df = df.rename(columns={df.columns[0]: 'participant'})
    col = {
        'tfd_pseudo':  get_col(df, 'Total Fixation Duration',  video, 'Rectangle_'),
        'tfd_map':     get_col(df, 'Total Fixation Duration',  video, 'Rectangle 2_'),
        'fc_pseudo':   get_col(df, 'Fixation Count',           video, 'Rectangle_'),
        'fc_map':      get_col(df, 'Fixation Count',           video, 'Rectangle 2_'),
        'ttff_pseudo': get_col(df, 'Time to First Fixation',   video, 'Rectangle_'),
        'ttff_map':    get_col(df, 'Time to First Fixation',   video, 'Rectangle 2_'),
        'fix_before':  get_col(df, 'Fixations Before',         video, 'Rectangle_'),
        'vc_pseudo':   get_col(df, 'Visit Count',              video, 'Rectangle_'),
        'vc_map':      get_col(df, 'Visit Count',              video, 'Rectangle 2_'),
        'ffd_pseudo':  get_col(df, 'First Fixation Duration',  video, 'Rectangle_'),
        'pct_pseudo':  get_col(df, 'Percentage Fixated',       video, 'Rectangle_'),
        'pct_map':     get_col(df, 'Percentage Fixated',       video, 'Rectangle 2_'),
    }
    for _, row in df.iterrows():
        pname = str(row.iloc[0]).strip()
        if pname.lower() in STAT_KW: continue
        pid = pname.split('-')[0].split('=')[0].strip()
        records.append({'participant': pid, 'algorithm': algo, 'group': grp,
            **{k: pd.to_numeric(row.get(v), errors='coerce') if v else np.nan
               for k, v in col.items()}})

df = pd.DataFrame(records)

# Derived metrics
df['ratio']          = df['tfd_pseudo'] / (df['tfd_map'] + 1e-9)
df['scanner_index']  = df['vc_pseudo']  / (df['tfd_pseudo'] + 1e-9)
df['avg_fix_depth']  = df['tfd_pseudo'] / (df['fc_pseudo']  + 1e-9)
df['switching_rate'] = (df['vc_pseudo'] + df['vc_map']) / (df['tfd_pseudo'] + df['tfd_map'] + 1e-9)
df['map_first']      = (df['ttff_map'] < df['ttff_pseudo']).astype(float)
df['algo_num']       = df['algorithm'].map({'BFS': 0, 'DFS': 1})  # ordinal for correlations

print(df.groupby(['group', 'algorithm']).size().unstack())
print(f'\nTotal: {len(df)} participants')

---
## 2. Group × Algorithm Descriptive Overview

In [ ]:
key_metrics = ['ratio', 'scanner_index', 'avg_fix_depth', 'switching_rate',
               'fix_before', 'ttff_pseudo', 'ffd_pseudo', 'tfd_pseudo', 'tfd_map']

summary = df.groupby(['group', 'algorithm'])[key_metrics].median().round(3)
print('=== Median values by Group × Algorithm ===')
print(summary.to_string())

In [ ]:
# Figure 1 — Heatmap of median values: rows = Group×Algo, cols = metrics
# Normalize each metric column to 0-1 for comparability
heat = summary.copy()
heat_norm = (heat - heat.min()) / (heat.max() - heat.min() + 1e-9)

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(
    heat_norm, annot=heat.values, fmt='.2f',
    cmap='YlOrRd', linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Normalized median (0=min, 1=max)'},
    annot_kws={'size': 8}
)
ax.set_title(
    'Figure 1. Median Eye-Tracking Metrics — Group × Algorithm\n'
    '(cell values = actual medians; color = normalized rank within each metric)',
    fontsize=11
)
ax.set_xlabel('')
ax.set_ylabel('Group  ×  Algorithm')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('exp_fig1_group_algo_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2 — 6-panel grid: one panel per metric, BFS vs DFS, each group as separate dot/line
focus_metrics = [
    ('ratio',          'Pseudocode/Map Ratio'),
    ('scanner_index',  'Scanner Index'),
    ('avg_fix_depth',  'Avg Fixation Depth (s)'),
    ('fix_before',     'Fixations Before Pseudocode'),
    ('ttff_pseudo',    'TTFF — Pseudocode (ms)'),
    ('switching_rate', 'Switching Rate'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(
    'Figure 2. Group × Algorithm Profiles — Metal Visualization\n'
    'Each line = one group; x-axis = BFS vs DFS; y-axis = median metric value',
    fontsize=12, fontweight='bold'
)

for ax, (col, label) in zip(axes.flat, focus_metrics):
    sub = df[df[col].replace([np.inf,-np.inf],np.nan).notna()].copy()
    if col == 'ratio': sub = sub[sub[col] < 25]

    medians = sub.groupby(['group','algorithm'])[col].median().reset_index()

    for grp in [1, 2, 3]:
        g = medians[medians['group']==grp].sort_values('algorithm')
        ax.plot(g['algorithm'], g[col], marker='o', linewidth=2, markersize=8,
                color=GRP_PAL[grp], label=GRP_LABEL[grp])

    # Raw points as jittered strip
    for i, algo in enumerate(['BFS','DFS']):
        for grp in [1,2,3]:
            vals = sub[(sub['algorithm']==algo)&(sub['group']==grp)][col].dropna()
            jitter = np.random.uniform(-0.08, 0.08, len(vals))
            ax.scatter(np.full(len(vals), i) + jitter, vals,
                       color=GRP_PAL[grp], alpha=0.25, s=20, zorder=1)

    ax.set_title(label, fontsize=10)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([0,1])
    ax.set_xticklabels(['BFS','DFS'])
    if col == focus_metrics[0][0]:
        ax.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('exp_fig2_group_profiles.png', bbox_inches='tight')
plt.show()

---
## 3. Group Effects — Kruskal-Wallis + Pairwise Tests

Does programming experience (Group 1 vs 2 vs 3) predict differences in attention metrics? Tested separately for BFS and DFS.

In [ ]:
test_metrics = [
    ('ratio',          'Pseudocode/Map Ratio'),
    ('scanner_index',  'Scanner Index'),
    ('avg_fix_depth',  'Avg Fixation Depth (s)'),
    ('switching_rate', 'Switching Rate'),
    ('fix_before',     'Fixations Before Pseudocode'),
    ('ttff_pseudo',    'TTFF — Pseudocode (ms)'),
    ('ffd_pseudo',     'First Fixation Duration (ms)'),
    ('tfd_pseudo',     'Total Fixation Duration — Pseudocode (s)'),
    ('tfd_map',        'Total Fixation Duration — Map (s)'),
]

print('=== Kruskal-Wallis: Group effect within each algorithm ===')
print(f'{"Metric":<40} {"BFS H":>7} {"BFS p":>8} {"DFS H":>7} {"DFS p":>8}')
print('-' * 75)

kw_results = []
for col, label in test_metrics:
    row = {'metric': col, 'label': label}
    for algo in ['BFS', 'DFS']:
        sub = df[df['algorithm']==algo]
        groups = [sub[sub['group']==g][col].replace([np.inf,-np.inf],np.nan).dropna()
                  for g in [1,2,3]]
        groups = [g for g in groups if len(g) >= 2]
        if len(groups) == 3:
            H, p = stats.kruskal(*groups)
            row[f'{algo}_H'] = H
            row[f'{algo}_p'] = p
        else:
            row[f'{algo}_H'] = row[f'{algo}_p'] = np.nan
    kw_results.append(row)
    sig_bfs = '***' if row.get('BFS_p',1)<.001 else '**' if row.get('BFS_p',1)<.01 else '*' if row.get('BFS_p',1)<.05 else '~' if row.get('BFS_p',1)<.10 else ''
    sig_dfs = '***' if row.get('DFS_p',1)<.001 else '**' if row.get('DFS_p',1)<.01 else '*' if row.get('DFS_p',1)<.05 else '~' if row.get('DFS_p',1)<.10 else ''
    print(f"{label:<40} {row.get('BFS_H',0):>7.2f} {row.get('BFS_p',1):>7.4f}{sig_bfs:<2} {row.get('DFS_H',0):>7.2f} {row.get('DFS_p',1):>7.4f}{sig_dfs}")

print('\nsig: ~ p<.10  * p<.05  ** p<.01  *** p<.001')

In [ ]:
# Pairwise Mann-Whitney for metrics with significant Kruskal-Wallis
print('=== Pairwise Group Comparisons (Mann-Whitney, two-sided) ===')
print('Only for metrics with KW p < 0.10 in either algorithm\n')

sig_metrics = [(r['metric'], r['label']) for r in kw_results
               if min(r.get('BFS_p', 1), r.get('DFS_p', 1)) < 0.10]

for col, label in sig_metrics:
    print(f'--- {label} ---')
    for algo in ['BFS', 'DFS']:
        sub = df[df['algorithm']==algo]
        print(f'  {algo}:')
        for g1, g2 in [(1,2),(1,3),(2,3)]:
            a = sub[sub['group']==g1][col].replace([np.inf,-np.inf],np.nan).dropna()
            b = sub[sub['group']==g2][col].replace([np.inf,-np.inf],np.nan).dropna()
            if len(a)<2 or len(b)<2: continue
            u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
            r = 1 - (2*u)/(len(a)*len(b))
            sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else '~' if p<.10 else 'ns'
            print(f'    G{g1} vs G{g2}: median {a.median():.3f} vs {b.median():.3f}  '
                  f'p={p:.4f} r={r:.3f} {sig}')
    print()

In [ ]:
# Figure 3 — Significant group-effect metrics: violin by group, faceted by algorithm
# Pick top 4 most significant
top_sig = sorted(sig_metrics, key=lambda x: min(
    next((r['BFS_p'] for r in kw_results if r['metric']==x[0]), 1),
    next((r['DFS_p'] for r in kw_results if r['metric']==x[0]), 1)
))[:4]

if top_sig:
    fig, axes = plt.subplots(len(top_sig), 2, figsize=(12, 4*len(top_sig)))
    if len(top_sig)==1: axes = [axes]
    fig.suptitle('Figure 3. Group Effects on Eye-Tracking Metrics (Metal)',
                 fontsize=12, fontweight='bold')

    group_order = [1, 2, 3]
    xtick_labels = ['G1\n(no exp)', 'G2\n(brief)', 'G3\n(years)']

    for row_axes, (col, label) in zip(axes, top_sig):
        for ax, algo in zip(row_axes, ['BFS', 'DFS']):
            sub = df[(df['algorithm']==algo) &
                     df[col].replace([np.inf,-np.inf],np.nan).notna()].copy()
            if col == 'ratio': sub = sub[sub[col]<25]

            sns.violinplot(data=sub, x='group', y=col, order=group_order,
                           palette=GRP_PAL_S, ax=ax, inner='quartile',
                           linewidth=1.1, cut=0)
            sns.stripplot(data=sub, x='group', y=col, order=group_order,
                          palette=GRP_PAL_S, ax=ax, size=4, alpha=0.65, jitter=True)

            ax.set_title(f'{label} — {algo}', fontsize=9)
            ax.set_xlabel('Group')
            ax.set_ylabel(label if algo=='BFS' else '')
            ax.set_xticklabels(xtick_labels)

    plt.tight_layout()
    plt.savefig('exp_fig3_group_effects.png', bbox_inches='tight')
    plt.show()
else:
    print('No significant group effects to plot.')

---
## 4. Group as Ordinal Predictor — Spearman Correlations

Group number (1, 2, 3) is treated as an ordinal variable representing programming experience. Spearman correlation between group and each metric reveals monotonic experience effects.

In [ ]:
print('=== Spearman Correlation: Group Number (1=novice → 3=expert) vs Metrics ===')
print(f'\n{"Metric":<40} {"BFS r":>7} {"BFS p":>8} {"DFS r":>7} {"DFS p":>8} {"ALL r":>7} {"ALL p":>8}')
print('-' * 85)

corr_metrics = [
    ('ratio',          'Pseudocode/Map Ratio'),
    ('scanner_index',  'Scanner Index'),
    ('avg_fix_depth',  'Avg Fixation Depth (s)'),
    ('switching_rate', 'Switching Rate'),
    ('fix_before',     'Fixations Before Pseudocode'),
    ('ttff_pseudo',    'TTFF — Pseudocode (ms)'),
    ('ffd_pseudo',     'First Fixation Duration (ms)'),
    ('tfd_pseudo',     'TFD — Pseudocode (s)'),
    ('tfd_map',        'TFD — Map (s)'),
    ('fc_pseudo',      'Fixation Count — Pseudocode'),
    ('fc_map',         'Fixation Count — Map'),
    ('vc_pseudo',      'Visit Count — Pseudocode'),
    ('vc_map',         'Visit Count — Map'),
    ('map_first',      'Map-First (bool)'),
]

grp_corr_rows = []
for col, label in corr_metrics:
    row_out = {'metric': col, 'label': label}
    for subset_name, subset in [('BFS', df[df['algorithm']=='BFS']),
                                  ('DFS', df[df['algorithm']=='DFS']),
                                  ('ALL', df)]:
        x = subset['group']
        y = subset[col].replace([np.inf,-np.inf], np.nan)
        valid = x.notna() & y.notna()
        if valid.sum() >= 5:
            r, p = stats.spearmanr(x[valid], y[valid])
            row_out[f'{subset_name}_r'] = r
            row_out[f'{subset_name}_p'] = p
        else:
            row_out[f'{subset_name}_r'] = row_out[f'{subset_name}_p'] = np.nan
    grp_corr_rows.append(row_out)

    def fmt(r, p):
        if np.isnan(r): return '    —       —'
        sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else '~' if p<.10 else ''
        return f'{r:>7.3f} {p:>7.4f}{sig:<3}'

    print(f"{label:<40} {fmt(row_out.get('BFS_r',np.nan), row_out.get('BFS_p',1))} "
          f"{fmt(row_out.get('DFS_r',np.nan), row_out.get('DFS_p',1))} "
          f"{fmt(row_out.get('ALL_r',np.nan), row_out.get('ALL_p',1))}")

print('\nsig: ~ p<.10  * p<.05  ** p<.01  *** p<.001')

In [ ]:
# Figure 4 — Bar chart of group→metric Spearman r values, BFS vs DFS side by side
gcr = pd.DataFrame(grp_corr_rows).set_index('metric')
labels_list = [r['label'] for r in grp_corr_rows]

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(labels_list))
w = 0.35

bfs_r = gcr['BFS_r'].values
dfs_r = gcr['DFS_r'].values
bfs_p = gcr['BFS_p'].values
dfs_p = gcr['DFS_p'].values

bars_b = ax.bar(x - w/2, bfs_r, w, color=ALG_PAL['BFS'], alpha=0.8, label='BFS')
bars_d = ax.bar(x + w/2, dfs_r, w, color=ALG_PAL['DFS'], alpha=0.8, label='DFS')

# Significance stars
for i, (rb, pb, rd, pd_) in enumerate(zip(bfs_r, bfs_p, dfs_r, dfs_p)):
    if not np.isnan(pb) and pb < 0.05:
        ax.text(i - w/2, rb + (0.02 if rb >= 0 else -0.05), '*', ha='center', fontsize=11)
    if not np.isnan(pd_) and pd_ < 0.05:
        ax.text(i + w/2, rd + (0.02 if rd >= 0 else -0.05), '*', ha='center', fontsize=11)

ax.axhline(0, color='#333', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels_list, rotation=40, ha='right', fontsize=9)
ax.set_ylabel('Spearman r  (group 1→3 = no exp → expert)', fontsize=10)
ax.set_title(
    'Figure 4. Spearman Correlation: Programming Experience (Group) vs Each Metric\n'
    '(* = p<.05; positive bar = metric increases with experience)',
    fontsize=11
)
ax.legend(fontsize=10)
ax.set_ylim(-0.6, 0.6)
plt.tight_layout()
plt.savefig('exp_fig4_group_correlations.png', bbox_inches='tight')
plt.show()

---
## 5. Full Pairwise Correlation Exploration

Every metric pair, Spearman r, for BFS and DFS separately. This table surfaces relationships not included in the primary analysis.

In [ ]:
all_metrics = [
    'tfd_pseudo', 'tfd_map', 'fc_pseudo', 'fc_map',
    'vc_pseudo', 'vc_map', 'ttff_pseudo', 'ttff_map',
    'fix_before', 'ffd_pseudo', 'pct_pseudo', 'pct_map',
    'ratio', 'scanner_index', 'avg_fix_depth', 'switching_rate', 'map_first', 'group'
]

pair_rows = []
for x_col, y_col in combinations(all_metrics, 2):
    for algo in ['BFS', 'DFS', 'ALL']:
        subset = df if algo == 'ALL' else df[df['algorithm']==algo]
        x = subset[x_col].replace([np.inf,-np.inf],np.nan)
        y = subset[y_col].replace([np.inf,-np.inf],np.nan)
        # Clip ratio outliers
        if x_col == 'ratio': x = x.where(x < 25)
        if y_col == 'ratio': y = y.where(y < 25)
        valid = x.notna() & y.notna()
        n = valid.sum()
        if n < 5: continue
        r, p = stats.spearmanr(x[valid], y[valid])
        pair_rows.append({'x': x_col, 'y': y_col, 'algo': algo,
                          'r': r, 'p': p, 'n': n, 'abs_r': abs(r)})

pairs_df = pd.DataFrame(pair_rows)

# Show top 30 strongest correlations (excluding trivial self-derived pairs)
trivial = {
    frozenset({'tfd_pseudo','ratio'}), frozenset({'tfd_map','ratio'}),
    frozenset({'vc_pseudo','scanner_index'}), frozenset({'tfd_pseudo','scanner_index'}),
    frozenset({'tfd_pseudo','avg_fix_depth'}), frozenset({'fc_pseudo','avg_fix_depth'}),
    frozenset({'vc_pseudo','switching_rate'}), frozenset({'vc_map','switching_rate'}),
    frozenset({'pct_pseudo','tfd_pseudo'}), frozenset({'pct_map','tfd_map'}),
    frozenset({'fc_pseudo','tfd_pseudo'}), frozenset({'fc_map','tfd_map'}),
    frozenset({'vc_pseudo','tfd_pseudo'}), frozenset({'vc_map','tfd_map'}),
}

non_trivial = pairs_df[
    pairs_df.apply(lambda r: frozenset({r['x'], r['y']}) not in trivial, axis=1)
]

top30 = (
    non_trivial
    .sort_values('abs_r', ascending=False)
    .drop_duplicates(subset=['x','y','algo'])
    .head(40)
[['x','y','algo','r','p','n']]
)

top30['sig'] = top30['p'].apply(
    lambda p: '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else '~' if p<.10 else ''
)
top30['r'] = top30['r'].round(3)
top30['p'] = top30['p'].round(4)

print('=== Top 40 Non-Trivial Pairwise Correlations (Spearman, sorted by |r|) ===')
print(top30.to_string(index=False))

In [ ]:
# Figure 5 — Full pairwise correlation matrix, BFS vs DFS side by side
display_metrics = [
    'ratio', 'scanner_index', 'avg_fix_depth', 'switching_rate',
    'tfd_pseudo', 'tfd_map', 'fc_pseudo', 'fc_map',
    'vc_pseudo', 'vc_map', 'fix_before', 'ttff_pseudo',
    'ttff_map', 'ffd_pseudo', 'map_first', 'group'
]
display_labels = [
    'ratio', 'scanner idx', 'fix depth', 'switching',
    'TFD pseudo', 'TFD map', 'FC pseudo', 'FC map',
    'VC pseudo', 'VC map', 'fix before', 'TTFF pseudo',
    'TTFF map', 'FFD pseudo', 'map first', 'group'
]

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle(
    'Figure 5. Full Spearman Correlation Matrix — Metal Visualization\n'
    'Left = BFS, Right = DFS  (ratio clipped at 25 for outlier robustness)',
    fontsize=12, fontweight='bold'
)

for ax, algo in zip(axes, ['BFS', 'DFS']):
    sub = df[df['algorithm']==algo][display_metrics].copy()
    sub['ratio'] = sub['ratio'].where(sub['ratio'] < 25)
    sub = sub.replace([np.inf,-np.inf], np.nan)
    corr = sub.corr(method='spearman')
    corr.index = display_labels
    corr.columns = display_labels
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr, mask=mask, annot=True, fmt='.2f',
        cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.6},
        annot_kws={'size': 6}
    )
    ax.set_title(f'Metal — {algo}', fontsize=11)

plt.tight_layout()
plt.savefig('exp_fig5_full_corr_matrix.png', bbox_inches='tight')
plt.show()

---
## 6. Deep Dives: Interesting Specific Correlations

Scatter plots for the most substantively interesting non-obvious pairs, broken down by group and algorithm.

In [ ]:
# --- FC pseudo vs FC map: trade-off or parallel engagement? ---
# If negative: viewers who fixate the code a lot fixate the map less (zero-sum attention)
# If positive: high engagement on both (some viewers are globally attentive)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    'Figure 6. Fixation Count: Pseudocode vs Map\n'
    'Trade-off (negative) or parallel engagement (positive)?',
    fontsize=11, fontweight='bold'
)

for ax, algo in zip(axes, ['BFS', 'DFS']):
    sub = df[df['algorithm']==algo].dropna(subset=['fc_pseudo','fc_map'])
    for grp in [1,2,3]:
        g = sub[sub['group']==grp]
        ax.scatter(g['fc_pseudo'], g['fc_map'], color=GRP_PAL[grp],
                   s=55, alpha=0.75, label=GRP_LABEL[grp], edgecolors='white', linewidth=0.3)
    m, b_ = np.polyfit(sub['fc_pseudo'], sub['fc_map'], 1)
    xl = np.linspace(sub['fc_pseudo'].min(), sub['fc_pseudo'].max(), 100)
    ax.plot(xl, m*xl + b_, color='#333', linewidth=1.5, linestyle='--', alpha=0.7)
    r, p = stats.spearmanr(sub['fc_pseudo'], sub['fc_map'])
    ax.set_title(f'{algo}  Spearman r = {r:.2f}, p = {p:.3f}', fontsize=10)
    ax.set_xlabel('Fixation Count — Pseudocode', fontsize=10)
    ax.set_ylabel('Fixation Count — Map', fontsize=10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('exp_fig6_fc_tradeoff.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- TTFF map vs ratio: does getting pulled into the map early predict lower code ratio? ---
# This is the 'first window hypothesis': the visualization has a critical early capture window.
# If early map grab → lower ratio: the opening seconds decide the whole trial.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    'Figure 7. Time-to-First-Fixation on Map vs Pseudocode/Map Ratio\n'
    'Longer TTFF map = viewer resisted the map initially — does that mean they read more code?',
    fontsize=11, fontweight='bold'
)

for ax, algo in zip(axes, ['BFS', 'DFS']):
    sub = df[(df['algorithm']==algo) & (df['ratio']<25)].dropna(subset=['ttff_map','ratio'])
    for grp in [1,2,3]:
        g = sub[sub['group']==grp]
        ax.scatter(g['ttff_map'], g['ratio'], color=GRP_PAL[grp],
                   s=55, alpha=0.75, label=GRP_LABEL[grp], edgecolors='white', linewidth=0.3)
    m, b_ = np.polyfit(sub['ttff_map'], sub['ratio'], 1)
    xl = np.linspace(sub['ttff_map'].min(), sub['ttff_map'].max(), 100)
    ax.plot(xl, m*xl + b_, color='#333', linewidth=1.5, linestyle='--', alpha=0.7)
    r, p = stats.spearmanr(sub['ttff_map'], sub['ratio'])
    ax.set_title(f'{algo}  Spearman r = {r:.2f}, p = {p:.3f}', fontsize=10)
    ax.set_xlabel('Time to First Fixation — Map (ms)', fontsize=10)
    ax.set_ylabel('Pseudocode/Map TFD Ratio', fontsize=10)
    ax.axhline(1.0, color='#888', linestyle=':', linewidth=1)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('exp_fig7_ttff_map_vs_ratio.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- fix_before vs avg_fix_depth: compensation hypothesis ---
# Do viewers who delay reading the code eventually read it more deeply (compensation)?
# Or do early map-watchers remain shallow code-readers (compounding deficit)?

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    'Figure 8. Fixations Before Pseudocode vs Avg Fixation Depth on Pseudocode\n'
    'Compensation (positive): late readers go deep. Compounding deficit (negative): they stay shallow.',
    fontsize=11, fontweight='bold'
)

for ax, algo in zip(axes, ['BFS', 'DFS']):
    sub = df[df['algorithm']==algo].dropna(subset=['fix_before','avg_fix_depth'])
    for grp in [1,2,3]:
        g = sub[sub['group']==grp]
        ax.scatter(g['fix_before'], g['avg_fix_depth'], color=GRP_PAL[grp],
                   s=55, alpha=0.75, label=GRP_LABEL[grp], edgecolors='white', linewidth=0.3)
    m, b_ = np.polyfit(sub['fix_before'], sub['avg_fix_depth'], 1)
    xl = np.linspace(sub['fix_before'].min(), sub['fix_before'].max(), 100)
    ax.plot(xl, m*xl + b_, color='#333', linewidth=1.5, linestyle='--', alpha=0.7)
    r, p = stats.spearmanr(sub['fix_before'], sub['avg_fix_depth'])
    ax.set_title(f'{algo}  Spearman r = {r:.2f}, p = {p:.3f}', fontsize=10)
    ax.set_xlabel('Fixations Before First Pseudocode Look', fontsize=10)
    ax.set_ylabel('Avg Fixation Depth on Pseudocode (s)', fontsize=10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('exp_fig8_fix_before_depth.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- TTFF pseudocode vs avg_fix_depth: urgency vs depth trade-off? ---
# Viewers who look at code sooner may do so in brief bursts (verification).
# Viewers who wait might arrive with more deliberate intent and read more deeply.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    'Figure 9. Time to First Fixation on Pseudocode vs Avg Fixation Depth\n'
    'Do viewers who look at code sooner read it more shallowly?',
    fontsize=11, fontweight='bold'
)

for ax, algo in zip(axes, ['BFS', 'DFS']):
    sub = df[df['algorithm']==algo].dropna(subset=['ttff_pseudo','avg_fix_depth'])
    for grp in [1,2,3]:
        g = sub[sub['group']==grp]
        ax.scatter(g['ttff_pseudo'], g['avg_fix_depth'], color=GRP_PAL[grp],
                   s=55, alpha=0.75, label=GRP_LABEL[grp], edgecolors='white', linewidth=0.3)
    m, b_ = np.polyfit(sub['ttff_pseudo'], sub['avg_fix_depth'], 1)
    xl = np.linspace(sub['ttff_pseudo'].min(), sub['ttff_pseudo'].max(), 100)
    ax.plot(xl, m*xl + b_, color='#333', linewidth=1.5, linestyle='--', alpha=0.7)
    r, p = stats.spearmanr(sub['ttff_pseudo'], sub['avg_fix_depth'])
    ax.set_title(f'{algo}  Spearman r = {r:.2f}, p = {p:.3f}', fontsize=10)
    ax.set_xlabel('Time to First Fixation — Pseudocode (ms)', fontsize=10)
    ax.set_ylabel('Avg Fixation Depth on Pseudocode (s)', fontsize=10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('exp_fig9_ttff_vs_depth.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Group vs scanner_index, separately by algorithm ---
# Does expertise predict scanning behavior differently for BFS vs DFS?
# Experts may scan BFS pseudocode (verify quickly) but deep-read DFS (unfamiliar execution pattern).

print('=== Group vs Scanner Index, by Algorithm ===')
for algo in ['BFS', 'DFS']:
    sub = df[df['algorithm']==algo].dropna(subset=['scanner_index'])
    r, p = stats.spearmanr(sub['group'], sub['scanner_index'])
    print(f'  {algo}: Spearman r = {r:.3f}, p = {p:.4f}, n = {len(sub)}')
    for grp in [1,2,3]:
        vals = sub[sub['group']==grp]['scanner_index']
        print(f'    G{grp} median={vals.median():.3f}, mean={vals.mean():.3f}')

print()
print('=== Group vs Avg Fixation Depth, by Algorithm ===')
for algo in ['BFS', 'DFS']:
    sub = df[df['algorithm']==algo].dropna(subset=['avg_fix_depth'])
    r, p = stats.spearmanr(sub['group'], sub['avg_fix_depth'])
    print(f'  {algo}: Spearman r = {r:.3f}, p = {p:.4f}, n = {len(sub)}')
    for grp in [1,2,3]:
        vals = sub[sub['group']==grp]['avg_fix_depth']
        print(f'    G{grp} median={vals.median():.3f}, mean={vals.mean():.3f}')

In [ ]:
# Figure 10 — Scanner index and avg_fix_depth across groups × algorithms
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(
    'Figure 10. Gaze Style by Experience Group — Metal Visualization\n'
    'Top: Scanner Index (higher = more scanning). Bottom: Avg Fixation Depth (higher = deeper reads)',
    fontsize=12, fontweight='bold'
)

metrics_style = [
    ('scanner_index', 'Scanner Index\n(visits/s on pseudocode)'),
    ('avg_fix_depth', 'Avg Fixation Depth (s)\non Pseudocode'),
]

for row_idx, (col, label) in enumerate(metrics_style):
    for col_idx, algo in enumerate(['BFS', 'DFS']):
        ax = axes[row_idx][col_idx]
        sub = df[(df['algorithm']==algo)].dropna(subset=[col])
        sns.boxplot(data=sub, x='group', y=col, palette=GRP_PAL_S, ax=ax,
                    order=[1,2,3], width=0.45, linewidth=1.2)
        sns.stripplot(data=sub, x='group', y=col, palette=GRP_PAL_S, ax=ax,
                      order=[1,2,3], size=5, alpha=0.65, jitter=True)
        r, p = stats.spearmanr(sub['group'], sub[col])
        ax.set_title(f'{label.split(chr(10))[0]} — {algo}\nSpearman r={r:.2f}, p={p:.3f}', fontsize=9)
        ax.set_xlabel('Group (1=novice, 3=expert)')
        ax.set_ylabel(label if col_idx==0 else '')
        ax.set_xticklabels(['G1\n(no exp)', 'G2\n(brief)', 'G3\n(years)'])

    # Third column: BFS vs DFS overlay scatter
    ax = axes[row_idx][2]
    for algo in ['BFS', 'DFS']:
        sub = df[df['algorithm']==algo].dropna(subset=[col])
        gm = sub.groupby('group')[col].median()
        ax.plot(gm.index, gm.values, marker='o', linewidth=2, markersize=9,
                color=ALG_PAL[algo], label=algo)
        for grp in [1,2,3]:
            vals = sub[sub['group']==grp][col].dropna()
            jitter = np.random.uniform(-0.06,0.06,len(vals))
            ax.scatter(np.full(len(vals), grp)+jitter, vals,
                       color=ALG_PAL[algo], alpha=0.2, s=18)
    ax.set_title(f'{label.split(chr(10))[0]}\nBFS vs DFS by Group', fontsize=9)
    ax.set_xlabel('Group')
    ax.set_ylabel('')
    ax.set_xticks([1,2,3])
    ax.set_xticklabels(['G1\n(no exp)','G2\n(brief)','G3\n(years)'])
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('exp_fig10_gaze_style_by_group.png', bbox_inches='tight')
plt.show()

---
## 7. Algorithm × Group Interaction

Does the BFS vs DFS effect differ depending on programming experience? For example:
- Does the ratio difference between BFS and DFS collapse for experts (who know both algorithms) but remain large for novices?
- Does the fixation depth advantage of DFS (deeper individual reads) disappear for experienced programmers?

In [ ]:
int_metrics = [
    ('ratio',          'Pseudocode/Map Ratio'),
    ('avg_fix_depth',  'Avg Fixation Depth (s)'),
    ('fix_before',     'Fixations Before Pseudocode'),
    ('ttff_pseudo',    'TTFF — Pseudocode (ms)'),
    ('scanner_index',  'Scanner Index'),
    ('switching_rate', 'Switching Rate'),
]

print('=== BFS vs DFS Mann-Whitney U within each Group ===')
print('(Effect of algorithm, tested separately per group)')
print()

int_rows = []
for col, label in int_metrics:
    print(f'--- {label} ---')
    for grp in [1, 2, 3]:
        b = df[(df['algorithm']=='BFS')&(df['group']==grp)][col]
        d = df[(df['algorithm']=='DFS')&(df['group']==grp)][col]
        if col == 'ratio':
            b = b.where(b<25); d = d.where(d<25)
        b = b.replace([np.inf,-np.inf],np.nan).dropna()
        d = d.replace([np.inf,-np.inf],np.nan).dropna()
        if len(b)<2 or len(d)<2: continue
        u, p = stats.mannwhitneyu(b, d, alternative='two-sided')
        r_rb = 1 - (2*u)/(len(b)*len(d))
        sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else '~' if p<.10 else 'ns'
        print(f'  G{grp}: BFS={b.median():.3f} DFS={d.median():.3f}  p={p:.4f}  r={r_rb:.3f}  {sig}')
        int_rows.append({'metric': col, 'label': label, 'group': grp,
                         'bfs_med': b.median(), 'dfs_med': d.median(),
                         'U': u, 'p': p, 'r': r_rb, 'sig': sig})
    print()

In [ ]:
# Figure 11 — Interaction heatmap: p-values and effect sizes for BFS vs DFS per group
int_df = pd.DataFrame(int_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    'Figure 11. Algorithm × Group Interaction — Does BFS vs DFS Differ Across Experience Levels?\n'
    'Left: rank-biserial r (effect size). Right: –log10(p) (significance). Metal visualization.',
    fontsize=11, fontweight='bold'
)

for ax, (val_col, fmt, title, cmap) in zip(axes, [
    ('r',   '.2f', 'Effect Size (rank-biserial r)', 'RdBu_r'),
    ('p',   '.3f', '–log10(p)',                     'YlOrRd'),
]):
    pivot = int_df.pivot(index='label', columns='group', values=val_col)
    if val_col == 'p':
        annot = pivot.map(lambda v: f'{v:.3f}' if not pd.isna(v) else '')
        plot_val = -np.log10(pivot.clip(lower=1e-10))
        center = None
        vmin, vmax = 0, 4
    else:
        annot = pivot.map(lambda v: f'{v:.2f}' if not pd.isna(v) else '')
        plot_val = pivot
        center = 0
        vmin, vmax = -1, 1

    plot_val.columns = ['G1 (no exp)', 'G2 (brief)', 'G3 (years)']
    annot.columns = plot_val.columns

    sns.heatmap(
        plot_val, annot=annot, fmt='', cmap=cmap, center=center,
        vmin=vmin, vmax=vmax, linewidths=0.5, ax=ax,
        cbar_kws={'shrink': 0.7}, annot_kws={'size': 9}
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Group')
    ax.set_ylabel('')

plt.tight_layout()
plt.savefig('exp_fig11_interaction_heatmap.png', bbox_inches='tight')
plt.show()

---
## 8. Within-Group Correlation Profiles

Key correlations run separately for each group and algorithm. Surfaces whether relationships change with experience — for example, whether the switching→depth trade-off is stronger or weaker for experts.

In [ ]:
focus_pairs = [
    ('fix_before',     'ratio',         'Fixations Before → Ratio'),
    ('switching_rate', 'avg_fix_depth', 'Switching → Fix Depth'),
    ('ttff_pseudo',    'avg_fix_depth', 'TTFF pseudo → Fix Depth'),
    ('ttff_map',       'ratio',         'TTFF map → Ratio'),
    ('fc_pseudo',      'fc_map',        'FC pseudo → FC map'),
    ('group',          'scanner_index', 'Group → Scanner Index'),
]

wg_rows = []
for x_col, y_col, pair_label in focus_pairs:
    for grp in [1, 2, 3]:
        for algo in ['BFS', 'DFS']:
            sub = df[(df['group']==grp)&(df['algorithm']==algo)]
            x = sub[x_col].replace([np.inf,-np.inf],np.nan)
            y = sub[y_col].replace([np.inf,-np.inf],np.nan)
            if y_col == 'ratio': y = y.where(y < 25)
            valid = x.notna() & y.notna()
            if valid.sum() < 4: continue
            r, p = stats.spearmanr(x[valid], y[valid])
            wg_rows.append({'pair': pair_label, 'group': grp, 'algo': algo,
                            'r': r, 'p': p, 'n': valid.sum()})

wg_df = pd.DataFrame(wg_rows)
print('=== Within-Group Correlation Profiles ===')
print(wg_df[['pair','group','algo','r','p','n']].round({'r':3,'p':4}).to_string(index=False))

In [ ]:
# Figure 12 — r value grid: rows=pair, cols=group, BFS/DFS side by side
pivot_bfs = wg_df[wg_df['algo']=='BFS'].pivot(index='pair', columns='group', values='r')
pivot_dfs = wg_df[wg_df['algo']=='DFS'].pivot(index='pair', columns='group', values='r')

for p in [pivot_bfs, pivot_dfs]:
    p.columns = ['G1 (no exp)', 'G2 (brief)', 'G3 (years)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    'Figure 12. Within-Group Spearman r for Key Correlation Pairs\n'
    'Left = BFS, Right = DFS. Dark red = strong positive; dark blue = strong negative.',
    fontsize=11, fontweight='bold'
)

for ax, (piv, algo) in zip(axes, [(pivot_bfs, 'BFS'), (pivot_dfs, 'DFS')]):
    # Get p-values for annotation
    p_piv = wg_df[wg_df['algo']==algo].pivot(index='pair', columns='group', values='p')
    p_piv.columns = piv.columns

    annot = piv.copy().astype(object)
    for col in piv.columns:
        for idx in piv.index:
            r_val = piv.loc[idx, col]
            p_val = p_piv.loc[idx, col] if idx in p_piv.index else 1.0
            if pd.isna(r_val):
                annot.loc[idx, col] = ''
            else:
                sig = '*' if p_val < 0.05 else '~' if p_val < 0.10 else ''
                annot.loc[idx, col] = f'{r_val:.2f}{sig}'

    sns.heatmap(
        piv.astype(float), annot=annot, fmt='',
        cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.7},
        annot_kws={'size': 10}
    )
    ax.set_title(f'Metal — {algo}', fontsize=11)
    ax.set_xlabel('Group')
    ax.set_ylabel('')

plt.tight_layout()
plt.savefig('exp_fig12_within_group_corr.png', bbox_inches='tight')
plt.show()

---
## 9. Inverted-U Test: Do Intermediates (Group 2) Show Peak Behavior?

Classic expertise reversal prediction: beginners under-engage (can't parse pseudocode), experts under-engage (don't need it), intermediates engage most because they're verifying a partially-formed mental model.

In [ ]:
# Test for quadratic (inverted-U) trend: fit group as 1/2/3 predictor of metric
# and test whether adding group^2 improves fit over linear alone

from scipy.stats import spearmanr

print('=== Inverted-U Check: Group Median by Metric ===')
print('Pattern of interest: G2 highest/lowest (inverted or regular U)')
print()

u_metrics = [
    ('ratio',          'Pseudocode/Map Ratio'),
    ('scanner_index',  'Scanner Index'),
    ('avg_fix_depth',  'Avg Fixation Depth (s)'),
    ('tfd_pseudo',     'TFD Pseudocode (s)'),
    ('fix_before',     'Fixations Before Pseudocode'),
]

for col, label in u_metrics:
    print(f'{label}:')
    for algo in ['BFS', 'DFS', 'ALL']:
        sub = df if algo == 'ALL' else df[df['algorithm']==algo]
        meds = []
        for grp in [1,2,3]:
            vals = sub[sub['group']==grp][col].replace([np.inf,-np.inf],np.nan).dropna()
            if col == 'ratio': vals = vals[vals<25]
            meds.append(vals.median())
        g2_is_peak   = meds[1] > meds[0] and meds[1] > meds[2]
        g2_is_trough = meds[1] < meds[0] and meds[1] < meds[2]
        pattern = ' ← G2 PEAK (inverted-U)' if g2_is_peak else ' ← G2 TROUGH (U-shape)' if g2_is_trough else ''
        print(f'  {algo:3}: G1={meds[0]:.3f}  G2={meds[1]:.3f}  G3={meds[2]:.3f}{pattern}')
    print()

In [ ]:
# Figure 13 — Inverted-U visualization: mean ± SE per group
fig, axes = plt.subplots(2, len(u_metrics), figsize=(5*len(u_metrics), 9))
fig.suptitle(
    'Figure 13. Group Trends — Testing for Inverted-U (Expertise Reversal) Pattern\n'
    'Error bars = ±1 SE. Inverted-U = G2 sits above both G1 and G3.',
    fontsize=12, fontweight='bold'
)

for col_idx, (col, label) in enumerate(u_metrics):
    for row_idx, algo in enumerate(['BFS', 'DFS']):
        ax = axes[row_idx][col_idx]
        sub = df[df['algorithm']==algo].copy()
        if col == 'ratio': sub = sub[sub[col]<25]
        sub = sub.dropna(subset=[col])

        grp_stats = sub.groupby('group')[col].agg(['mean','std','count']).reset_index()
        grp_stats['se'] = grp_stats['std'] / np.sqrt(grp_stats['count'])

        ax.errorbar(grp_stats['group'], grp_stats['mean'], yerr=grp_stats['se'],
                    marker='o', linewidth=2, markersize=9, capsize=4,
                    color=ALG_PAL[algo])

        # Raw strip
        for grp in [1,2,3]:
            vals = sub[sub['group']==grp][col].dropna()
            ax.scatter(np.full(len(vals), grp) + np.random.uniform(-0.1,0.1,len(vals)),
                       vals, color=ALG_PAL[algo], alpha=0.2, s=18, zorder=1)

        ax.set_title(f'{label}\n{algo}', fontsize=9)
        ax.set_xlabel('Group')
        ax.set_ylabel(label if col_idx==0 else '')
        ax.set_xticks([1,2,3])
        ax.set_xticklabels(['G1','G2','G3'])

plt.tight_layout()
plt.savefig('exp_fig13_inverted_u.png', bbox_inches='tight')
plt.show()

---
## 10. Interpretation Guide

### Group effects on attention style (Section 3 & 4)

**If scanner index increases with group (negative r with ratio):**  
Experts scan pseudocode rapidly — brief checks rather than sustained reads. Novices may read slowly and carefully, or may not read at all (and just watch the map). Intermediates are the likely deep readers, using the pseudocode to verify a partially-formed mental model.

**If avg_fix_depth decreases with group:**  
Consistent with expertise: experts require less fixation time per line because they recognize pseudocode structure immediately. Novices may stare longer at each line because it requires decoding effort.

**If ratio decreases with group:**  
Experts read less pseudocode relative to the map — they can infer algorithm execution from the graph alone. Novices rely more on the code. Design implication: a single pseudocode panel fails both endpoints — too hard to read for novices, unnecessary for experts.

---

### FC pseudo vs FC map trade-off (Section 6, Figure 6)

**If negative correlation:** Attention is zero-sum — viewers allocate fixations between code and map in trade-off. High code-fixation viewers are systematically under-fixating the map, and vice versa. This is a dual-representation problem: two panels compete for a fixed attentional budget.

**If positive correlation:** Some viewers are globally high-engagement (fixate everything more) while others are globally low-engagement (passive watchers). This is an individual engagement-level effect, not a code-vs-map trade-off. Design implication: the problem isn't competition between panels — it's activating globally passive viewers.

---

### TTFF map vs ratio (Section 6, Figure 7)

**If positive (longer TTFF on map → higher ratio):**  
Viewers who resist the map for longer accumulate more pseudocode reading. The longer they avoid the map, the more code they read. This is the "map trap" hypothesis: the animated graph is a gravitational pull, and viewers who don't get pulled in early orient toward code.

**Design implication:** Show only pseudocode for the first 2–3 seconds; delay map animation. If the map is not yet moving when the viewer's first fixation lands, they have no choice but to read code — and this starting position may carry through the trial.

---

### Fix before → avg_fix_depth: compensation vs compounding (Figure 8)

**If positive (more pre-pseudocode fixations → deeper eventual reads):**  
Viewers who delay reading the code eventually do so more deliberately. They've absorbed the visual context first and now read the code with purpose. This is a beneficial delay — watching the animation first may help them understand what the pseudocode is describing.

**If negative (more pre-pseudocode fixations → shallower reads):**  
Delay compounds into shallower engagement. Once the map captures attention, the viewer never fully commits to the code — their eventual pseudocode fixations are quick and shallow. This is the compounding deficit model.

---

### Algorithm × Group interaction (Section 7, Figure 11)

**If BFS/DFS effect is significant only in G1 or G2, not G3:**  
Experts are immune to the algorithm's visual complexity — they read the same way regardless of whether it's BFS or DFS because they already have strong prior schemas for both. Only learners without strong schemas are affected by the algorithm's traversal pattern.

**If effect is significant only in G3:**  
Experts are the ones whose attention allocation shifts with algorithm. Possible explanation: novices can't track either algorithm visually, so they default to code regardless. Experts can track DFS visually (and do), but not BFS's wave expansion — so their code reading is specifically elevated for BFS.

---

### Inverted-U (Section 9, Figure 13)

**If G2 peaks on ratio or tfd_pseudo:**  
Intermediates read pseudocode most. Novices can't parse it; experts don't need it. Intermediates are at the sweet spot of partial understanding — they know enough to use the code as a verification tool but not enough to skip it. This is a theoretically important result: the pseudocode panel is primarily valuable for one of three audience segments.

**If G2 peaks on scanner_index:**  
Intermediates scan most rapidly — checking and cross-referencing. Novices read slowly or not at all; experts read in single deep passes. The intermediate scanning pattern is the hallmark of active integration.

---

## 11. Limitations

1. **n ≈ 19–20 per group per algorithm.** Pairwise tests within group × algorithm cells have low power. Treat marginal p-values (p < .10) as directional only.
2. **Group self-selection.** Group assignment reflects pre-existing experience, not randomized treatment. Observed group differences are correlational.
3. **No comprehension test.** Differences in fixation patterns cannot be linked to learning outcomes without a knowledge measure.
4. **AOI naming inferred.** Rectangle = Pseudocode, Rectangle 2 = Map — verify against the Tobii Studio project.
5. **Metal only.** All results are conditional on Metal visualization style.